In [1]:
import os
import re
import csv
from collections import defaultdict

def parse_log_file_all_metrics(path):
    """从日志中解析所有 start to eval 段（自动提取 epoch）"""
    records = []
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if line.strip().startswith("start to eval"):
            # === 1. 向上查找最近的 "N val1 val2 val3" 行 ===
            epoch = -1
            for j in range(i - 1, max(-1, i - 50), -1):
                m = re.match(r"^\s*(\d+)\s+[-+]?\d+\.\d+", lines[j])
                if m:
                    epoch = int(m.group(1))
                    break

            eval_metrics = {}
            # === 2. 提取 hit 系列 ===
            if i + 1 < len(lines) and lines[i + 1].startswith("hit20"):
                m = re.search(r"hit20\s+([0-9.+-eE]+)", lines[i + 1])
                if m:
                    eval_metrics["hit20"] = float(m.group(1))
            if i + 2 < len(lines) and lines[i + 2].startswith("hit50"):
                m = re.search(r"hit50\s+([0-9.+-eE]+)", lines[i + 2])
                if m:
                    eval_metrics["hit50"] = float(m.group(1))
            if i + 3 < len(lines) and lines[i + 3].startswith("hit100"):
                m = re.search(r"hit100\s+([0-9.+-eE]+)", lines[i + 3])
                if m:
                    eval_metrics["hit100"] = float(m.group(1))

            # === 3. 提取复合指标 ===
            if i + 4 < len(lines) and lines[i + 4].startswith("roc_auc"):
                nums = re.findall(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", lines[i + 4])
                if len(nums) >= 5:
                    fields = ['roc_auc', 'pr_auc', 'f1', 'mrr_pess', 'mrr_opt']
                    eval_metrics.update(dict(zip(fields, map(float, nums[-5:]))))

            if eval_metrics:
                eval_metrics["epoch"] = epoch
                records.append(eval_metrics)
    return records


# ==== 参数 ====
dataset = "ogbl_citation2"
ratio = 0.02
cs = [1]
gammas = [2.5, 2.75, 3, 3.25, 3.5]
alphas = [1, 2.5, 3, 3.5, 5, 8]
seeds = [1, 2, 3, 4, 5]

log_base_dir = "."
output_dir = "./results"
os.makedirs(output_dir, exist_ok=True)

ALL_METRICS = ["epoch", "hit20", "hit50", "hit100", "roc_auc", "pr_auc", "f1", "mrr_pess", "mrr_opt"]

# ==== 主逻辑 ====
rows = []
for c in cs:
    for gamma in gammas:
        for alpha in alphas:
            for seed in seeds:
                fname = f"g{gamma}-a{alpha}-s{seed}-r{ratio}.log"
                log_path = os.path.join(log_base_dir, fname)
                if not os.path.isfile(log_path):
                    print(f"[WARN] 缺失: {log_path}")
                    row = {"c": c, "gamma": gamma, "alpha": alpha, "seed": seed, "epoch": -1}
                    for k in ALL_METRICS:
                        if k not in ["epoch"]:
                            row[k] = 0.0
                    rows.append(row)
                    continue

                records = parse_log_file_all_metrics(log_path)
                if not records:
                    print(f"[WARN] 无有效指标: {log_path}")
                    row = {"c": c, "gamma": gamma, "alpha": alpha, "seed": seed, "epoch": -1}
                    for k in ALL_METRICS:
                        if k not in ["epoch"]:
                            row[k] = 0.0
                    rows.append(row)
                    continue

                for rec in records:
                    row = {"c": c, "gamma": gamma, "alpha": alpha, "seed": seed, "epoch": rec.get("epoch", -1)}
                    for k in ALL_METRICS:
                        if k not in ["epoch"]:
                            row[k] = rec.get(k, 0.0) * 100
                    rows.append(row)

# ==== 写入 CSV ====
csv_path = os.path.join(output_dir, f"{dataset}_epochwise_result.csv")
header = ["c", "gamma", "alpha", "seed"] + ALL_METRICS
with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()
    writer.writerows(rows)

print(f"[INFO] 共解析 {len(rows)} 条记录，写入完毕: {csv_path}")


[INFO] 共解析 1500 条记录，写入完毕: ./results/ogbl_citation2_epochwise_result.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact
import numpy as np

# === 读取数据 ===
csv_path = "./results/ogbl_citation2_epochwise_result.csv"
df = pd.read_csv(csv_path)
df["mrr_pess"] = df["mrr_pess"].astype(float)
df["epoch"] = df["epoch"].astype(int)

# === 对 seed 聚合 ===
agg_df = (
    df.groupby(["gamma", "alpha", "epoch"], as_index=False)
      .agg(mean_mrr_pess=("mrr_pess", "mean"))
)

# === 交互式绘图函数 ===
def plot_epoch(epoch=0, ylim_min=None, ylim_max=None):
    sns.set(style="whitegrid")
    fig, ax = plt.subplots(figsize=(8, 6))

    data = agg_df[agg_df["epoch"] == epoch]
    if data.empty:
        print(f"epoch={epoch} 数据为空。")
        return

    gammas = sorted(data["gamma"].unique())
    palette = sns.color_palette("Blues", n_colors=len(gammas))

    ymin_auto = data["mean_mrr_pess"].replace(0, np.nan).min()
    ymax_auto = data["mean_mrr_pess"].replace(0, np.nan).max()
    if pd.isna(ymin_auto) or pd.isna(ymax_auto):
        print("数据全为 0。")
        return

    ymin = ylim_min if ylim_min is not None else ymin_auto
    ymax = ylim_max if ylim_max is not None else ymax_auto

    for i, gamma in enumerate(gammas):
        sub = data[data["gamma"] == gamma].sort_values("alpha")
        ax.plot(
            sub["alpha"], sub["mean_mrr_pess"],
            marker="o", color=palette[i], label=f"γ={gamma}"
        )

    ax.set_title(f"mrr_pess vs alpha (epoch={epoch})")
    ax.set_xlabel("alpha")
    ax.set_ylabel("mean_mrr_pess")
    ax.set_ylim(ymin, ymax)
    ax.legend(title="gamma")
    ax.grid(True)
    plt.tight_layout()
    plt.show()

# === 交互式控件 ===
# === 交互式控件 ===
epoch_options = sorted(df["epoch"].unique())
if len(epoch_options) == 0:
    print("未检测到任何 epoch 值，请检查输入文件。")
else:
    default_epoch = epoch_options[0]  # ✅ 自动选择第一个 epoch
    interact(
        plot_epoch,
        epoch=widgets.Dropdown(options=epoch_options, value=default_epoch, description="Epoch"),
        ylim_min=widgets.FloatText(value=None, description="ymin (可留空)"),
        ylim_max=widgets.FloatText(value=None, description="ymax (可留空)")
    )



interactive(children=(Dropdown(description='Epoch', options=(4, 9, 14, 19, 24, 29, 34, 39, 44, 49), value=4), …